**Vision-Based Gaze Estimation | Presented by: Reza Saadatyar (2026)**

**01 | MPIIGaze Source Audit and Participant-Level Protocol**

![MPIIGaze archive structure, recorded metadata and participant-level evaluation design](../../images/01_mpiigaze_source_audit_and_protocol.svg)

**Objective:** Inspect source files, participant/day MATLAB structure and evaluation design for unseen-person gaze estimation.

**Dataset:** [MPIIGaze](https://www.collaborative-ai.org/research/datasets/MPIIGaze/) / [DaRUS archive](https://doi.org/10.18419/darus-3230). The authors report 213,659 collected images from 15 people. The recorded archive audit identified 430,180 file paths and 15 participant IDs.

**Scope:** Source inventory, participant support, representative MATLAB headers and publisher list metadata. Notebook 02 examines image–target alignment and the normalised gaze coordinate frame.

**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

/content/Eye-Tracking-Neurocognitive-Intelligence


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("vision_based_gaze_estimation")

# ---------------------------- 2. Standard Library ----------------------------
import shutil
import sys

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

# ----------------- 5. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from mpiigaze_io import (
    SOURCE_PAGE, SOURCE_DOI, ensure_mpiigaze_source,
    source_inventory, participant_source_support, inspect_mat_header,
    is_archive_companion, is_normalised_mat_path,
)

# ------------------------ 6. Output and Source Paths -------------------------
study = root / "vision_based_gaze_estimation"
output_dir = study / "output"
output_dir.mkdir(parents=True, exist_ok=True)
raw_dir = root / "data" / "raw" / "mpiigaze"
raw_dir.parent.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4),
)
print("Study outputs:", output_dir)
print("Source directory:", raw_dir)
print("Python:", sys.version.split()[0])
print("Free disk (GiB):", round(shutil.disk_usage(raw_dir.parent).free / 2**30, 2))


**02. Locate or Acquire the MPIIGaze Source**

MPIIGaze is available for non-commercial research. The source adapter reuses extracted data or a cached archive; otherwise it downloads and extracts the official distribution in the runtime. Raw data remain in `data/raw/mpiigaze/`.

In [ ]:
# ==================== 02. Locate Official MPIIGaze Source ====================

# --------------------- 1. Acquire or Reuse Original Data ---------------------
raw_dir = ensure_mpiigaze_source(raw_dir)

# ------------------------- 2. Report Runtime Source --------------------------
print("Source directory:", raw_dir)
print("Official source:", SOURCE_PAGE)
print("Archive reference:", SOURCE_DOI)

**03. Audit Original Source Files**

The inventory retains archive paths, component names, participant IDs and file sizes. Operating-system companions are separated from scientific files. A MATLAB container can hold multiple image and target records.

In [ ]:
# =================== 03. Audit Source Files and Provenance ===================

# ---------------------- 1. Build the Observed Inventory ----------------------
inventory = source_inventory(raw_dir)
if inventory.empty or inventory.relative_path.duplicated().any():
    raise ValueError("Source files missing or duplicated")
inventory.to_csv(output_dir / "01_file_inventory.csv", index=False)

# ---------------------- 2. Separate Archive Companions -----------------------
companion = inventory.relative_path.map(is_archive_companion)
scientific = inventory.loc[~companion]
normalized = inventory.loc[
    inventory.relative_path.map(is_normalised_mat_path)
].sort_values(["participant_id", "relative_path"])
if normalized.empty:
    raise ValueError("No participant/day normalised MAT files found")
print("Archive file paths:", len(inventory))
print("Archive companion paths:", int(companion.sum()))
print("Participant/day MAT paths:", len(normalized))
display(scientific.groupby("component").agg(
    files=("relative_path", "size"),
    mebibytes=("size_bytes", lambda x: round(x.sum() / 2**20, 1)),
).reset_index())
display(inventory.head(12))

**04. Audit Participant and Day Support**

Summarise participant IDs and normalised day containers. Archive path counts include companion entries; `normalised_mat_files` counts only participant/day MAT paths. Neither count establishes the usable image–target sample size.

In [ ]:
# =================== 04. Audit Participant and Day Support ===================

# ----------------- 1. Summarise Participant Source Availability --------------
support = participant_source_support(inventory)
if support.empty or support.participant_id.duplicated().any():
    raise ValueError("No participant folder identifiers found")
support.to_csv(output_dir / "01_participant_support.csv", index=False)
display(support)
print("Participants with normalised MAT paths:",
      support.normalised_mat_files.gt(0).sum())

**05. Inspect Normalised MATLAB Schemas**

Inspect one genuine day container per participant. The recorded representative headers contain a `data` structure and a `filenames` cell array. Nested image and gaze fields require inspection in Notebook 02.

In [ ]:
# =================== 05. Inspect Normalised MATLAB Schemas ===================

# ------------------ 1. Select One Container per Participant ------------------
representatives = normalized.drop_duplicates("participant_id")
schema_rows = []
for item in representatives.itertuples(index=False):
    for variable in inspect_mat_header(raw_dir / item.relative_path):
        schema_rows.append({
            "participant_id": item.participant_id,
            "relative_path": item.relative_path, **variable
        })
mat_schema = pd.DataFrame(schema_rows)
if mat_schema.empty:
    raise ValueError("No MATLAB header variables were found")
mat_schema.to_csv(output_dir / "01_mat_schema.csv", index=False)
print("MAT containers inspected:", len(representatives))
display(mat_schema.head(30))

**06. Inspect Evaluation and Annotation Lists**

Inspect publisher text files, excluding archive companions and identifying documentation separately from participant lists. Non-empty text lines are not verified unique gaze samples.

In [ ]:
# ================ 06. Examine Evaluation and Annotation Lists ================

# -------------------- 1. Identify Publisher Sample Lists ---------------------
list_files = scientific.loc[
    scientific.component.isin(["Evaluation Subset", "Annotation Subset"])
    & scientific.suffix.isin([".txt", ".csv", ".list"])
]
list_rows = []
for item in list_files.itertuples(index=False):
    with (raw_dir / item.relative_path).open(
            encoding="utf-8", errors="replace") as handle:
        count = sum(bool(line.strip()) for line in handle)
    file_type = ("documentation" if Path(item.relative_path).stem.lower()
                 .startswith("readme") else "participant_list")
    list_rows.append({
        "relative_path": item.relative_path,
        "component": item.component,
        "file_type": file_type,
        "nonempty_text_lines": count,
    })
lists = pd.DataFrame(list_rows, columns=[
    "relative_path", "component", "file_type", "nonempty_text_lines"])
lists.to_csv(output_dir / "01_evaluation_list_inventory.csv", index=False)
display(lists.groupby(["component", "file_type"]).size().rename("files"))
display(lists.head(30))

**07. Record a Participant-Level Protocol**

Leave-one-person-out (LOPO) evaluation holds out one participant in each outer fold. Two distinct inner Validation participants support model selection. The primary metric is 3D angular error in degrees. Notebook 02 constructs actual fold assignments from eligible samples.

In [ ]:
# =================== 07. Record Participant-Level Protocol ===================

# ----------------------- 1. Save Evaluation Settings -------------------------
protocol = {
    "dataset": "MPIIGaze",
    "source_url": SOURCE_PAGE,
    "dataset_doi": SOURCE_DOI,
    "outer_protocol": "participant-disjoint LOPO",
    "inner_protocol": "participant-disjoint validation",
    "inner_validation_people": 2,
    "primary_metric": "3D angular error in degrees",
}
pd.DataFrame(protocol.items(), columns=["setting", "value"]).to_csv(
    output_dir / "01_protocol.csv", index=False)
display(pd.DataFrame(protocol.items(), columns=["setting", "value"]))

**08. Plot Source Inventory**

A logarithmic file-count axis makes small and large components visible together. Exact counts are annotated; a separate panel shows participant/day MAT support. These counts describe source files, not model results.

In [ ]:
# ==================== 08. Plot Observed Source Inventory =====================

# ------------------------ 1. Source Component Figure -------------------------
counts = scientific.component.value_counts().sort_values()
mat_candidates = inventory.loc[
    inventory.component.eq("Normalized") & inventory.suffix.eq(".mat")]
excluded = int(mat_candidates.relative_path.map(is_archive_companion).sum())
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8),
                         gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
bars = ax.barh(counts.index, counts.values)
ax.set_xscale("log")
ax.set_xlim(0.8, max(counts.max() * 6, 10))
ax.bar_label(bars, labels=[f"{n:,}" for n in counts], padding=4)
ax.set(xlabel="Non-companion source paths (log scale)",
       title="Source components")
ax.grid(axis="x", alpha=.2)
ax = axes[1]
bars = ax.bar(support.participant_id, support.normalised_mat_files)
ax.bar_label(bars, padding=2, fontsize=8)
ax.set(ylabel="Participant/day MAT paths", title="Normalised MAT support")
ax.set_ylim(0, max(1, support.normalised_mat_files.max() * 1.2))
ax.tick_params(axis="x", rotation=60)
fig.suptitle(f"MPIIGaze source audit | {len(normalized):,} day MAT paths; "
             f"{excluded:,} MAT companions excluded")
fig.tight_layout()
save_figure(fig, output_dir / "01_source_inventory.png")
plt.show()

**09. Interpretation and Output Download**

File inventory and header metadata describe source availability and structure. Notebook 02 examines sample correspondence and gaze geometry, then constructs participant-disjoint folds. Derived CSV tables and the source plot can be downloaded together.

In [ ]:
# ================= 09. Interpret Source Evidence and Export ==================

# --------------------- 1. Report Observed Source Support ---------------------
print("Archive paths:", len(inventory))
print("Participant/day MAT paths:", len(normalized))
print("MAT header variables:", len(mat_schema))
print("Participant list paths:", int(lists.file_type.eq("participant_list").sum()))

# -------------------------- 2. Package Current Outputs -----------------------
(output_dir / "01_protocol.json").unlink(missing_ok=True)
download_outputs_from_colab("vision_based_gaze_estimation/output", "01")